# Unified Kaggle Gate 1

New private notebook only. Select T4 (T4 x2 is acceptable; only GPU 0 is used), enable Internet, and attach the private `unified_gate1_20260928_v2.bin` dataset input. No API key or tunnel is needed. Optional: enable `HF_TOKEN` through Kaggle Secrets if download access requires it.

Run the following three code cells in order. This installs the candidate only in this new notebook and runs three fresh pipelines sequentially. No training, application migration or adapter switching. Return `evidence.zip` even when inference fails. Completion still requires output/resource review; it is not an automatic Gate 1 pass.

In [ ]:
from pathlib import Path
from hashlib import sha256
from zipfile import ZipFile
import json, shutil, sys, time

BUNDLE_NAME = "unified_gate1_20260928_v2.bin"
EXPECTED_SHA256 = "8ed784714d534ff767dd261bd5d440d643b526cfac60f2d8dccd47e2c9d44293"
ROOT = Path("/kaggle/working/gate1_bundle")
OUT = Path("/kaggle/working/unified_gate1")
started = time.monotonic()
free_before = shutil.disk_usage("/tmp").free
matches = list(Path("/kaggle/input").rglob(BUNDLE_NAME))
if len(matches) != 1:
    raise RuntimeError(f"Attach exactly one private dataset containing {BUNDLE_NAME}; found {len(matches)}")
bundle = matches[0]
h = sha256()
with bundle.open("rb") as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        h.update(chunk)
if h.hexdigest() != EXPECTED_SHA256:
    raise RuntimeError("Wrong or corrupted Gate 1 upload. Do not extract it.")
if ROOT.exists() or OUT.exists():
    raise RuntimeError("Prior evidence exists. Use a fresh session, or change ROOT and OUT to new paths.")
ROOT.mkdir(parents=True)
with ZipFile(bundle) as archive:
    if archive.testzip() is not None:
        raise RuntimeError("Bundle CRC failure")
    names = archive.namelist()
    if len(names) != len(set(names)):
        raise RuntimeError("Duplicate ZIP members")
    for name in names:
        target = (ROOT / name).resolve()
        if name.startswith("/") or "\\" in name or ":" in name or not target.is_relative_to(ROOT.resolve()):
            raise RuntimeError("Unsafe ZIP member")
    manifest = json.loads(archive.read("gate1_bundle.json"))
    if set(names) != set(manifest["files"]) | {"gate1_bundle.json"}:
        raise RuntimeError("Unexpected bundle members")
    for name, expected in manifest["files"].items():
        if sha256(archive.read(name)).hexdigest() != expected:
            raise RuntimeError(f"Member hash mismatch: {name}")
    archive.extractall(ROOT)
print("BUNDLE VERIFIED", bundle.name, bundle.stat().st_size, "bytes")
print("Free scratch before extraction:", free_before, "bytes")
print("Extraction/preflight:", round(time.monotonic() - started, 2), "seconds")


In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, str(ROOT / "scripts/unified_gate1_bootstrap.py"), "--output", str(OUT)],
    cwd=ROOT,
    check=False,
)
print("Bootstrap exit code:", result.returncode)
print("Run the next cell and return evidence even if this code is nonzero.")


In [ ]:
from IPython.display import FileLink, display

setup_file = OUT / "setup.json"
if setup_file.exists():
    setup = json.loads(setup_file.read_text())
    print("Setup status:", setup["status"], "phase:", setup["phase"])
    if setup.get("error"):
        print(setup["error"])
summary_file = OUT / "results/summary.json"
if summary_file.exists():
    summary = json.loads(summary_file.read_text())
    print("Probe status:", summary["status"])
    for feature in summary["results"]:
        print(feature["feature"], feature.get("style_id"), feature["status"],
              feature.get("failure_category"), feature.get("error"))
print("No automatic Gate 1 pass. Gate 2 has not started.")
evidence = OUT / "evidence.zip"
if evidence.exists():
    display(FileLink(str(evidence)))
else:
    print("No evidence ZIP. Send the cell error and setup/log files that exist.")
